# Projekt Lyckomaximering

Det är valår i Sverige. Medan de etablerade partierna debatterar utifrån gamla ideologier har ett nytt, datadrivet parti bestämt sig för att fokusera på en sak: **livskvalitet**.

Uppdraget är att analysera World Happiness Report 2019 och identifiera vilka faktorer som verkar hänga ihop med högre `Score`. Målet är att ge partiet ett underlag som kan hjälpa dem att formulera ett mer datadrivet partiprogram.

> Vi vill inte gissa vad folk mår bra av. Vi vill använda datareduktion och visualisering för att förstå vad som faktiskt kännetecknar lyckliga samhällen.

## Frågeställning och metod

Analysen utgår från sex faktorer i datasetet:

- `GDP per capita`
- `Social support`
- `Healthy life expectancy`
- `Freedom to make life choices`
- `Generosity`
- `Perceptions of corruption`

Vi använder först deskriptiv analys och visualisering för att förstå datan. Därefter jämför vi SVD med och utan standardisering, och avslutar med UMAP för att undersöka om länderna bildar tydliga grupper baserat på sina lyckoprofiler.

Datasetet innehåller inte enbart rådata, utan även bearbetade faktorvärden från World Happiness Report. Analysen undersöker därför hur dessa faktorbidrag samvarierar och grupperar länder, snarare än att direkt mäta kausala effekter från rå samhällsdata.

En viktig tolkningsnotering är att kolumnen `Perceptions of corruption` i denna CSV används som ett positivt faktorbidrag kopplat till låg uppfattad korruption. I visualiseringar beskrivs höga värden därför som **Perceived lack of corruption**.

## Variabelförklaringar

Källor: [Kaggle, World Happiness Report](https://www.kaggle.com/datasets/unsdsn/world-happiness?select=2019.csv) och [World Happiness Report 2019, Statistical Appendix 1](https://files.worldhappiness.report/WHR19_Statistical_Appendix_01.pdf).

Kolumnnamnen kommer från `2019.csv`, men featurekolumnerna efter `Score` är inte råvärden. De är faktorbidrag som World Happiness Report använder för att förklara skillnader i `Score` relativt Dystopia. Rapporten beskriver detta som *"components explained by six hypothesized underlying determinants"* och betonar att modellen är *"illustrative rather than conclusive"*.

| Variabel | Exaktare källgrund | Användning i analysen |
|---|---|---|
| `Overall rank` | WHR-tabellerna rangordnar länder efter *"happiness scores"*. | Rankning i `2019.csv`, baserad på `Score`. |
| `Country or region` | WHR använder land/territorium, alltså *"country/territory"*. | Land eller region i `2019.csv`. |
| `Score` | `Cantril Ladder`: respondenten placerar sitt liv på en stege från `0` till `10`. | Det övergripande måttet på livsutvärdering. |
| `GDP per capita` | Grundvariabeln bygger på `GDP per capita` i *"purchasing power parity"*. | I `2019.csv` är detta faktorbidraget från `GDP per capita`, inte rå BNP. |
| `Social support` | GWP-fråga om man har *"relatives or friends you can count on"* vid problem. | Faktorbidrag kopplat till `Social support`. |
| `Healthy life expectancy` | WHR använder *"Healthy life expectancies at birth"*. | Faktorbidrag kopplat till `Healthy life expectancy`. |
| `Freedom to make life choices` | GWP-fråga om man är nöjd med *"freedom to choose what you do with your life"*. | Faktorbidrag kopplat till `Freedom to make life choices`. |
| `Generosity` | Bygger på frågan om man har *"donated money to a charity"* och beräknas som en residual. | Faktorbidrag kopplat till `Generosity`, justerat mot `GDP per capita`. |
| `Perceptions of corruption` | Grundvariabeln är medelvärdet av två GWP-frågor: *"Is corruption widespread throughout the government or not?"* och *"Is corruption widespread within businesses or not?"*. Om government-data saknas används business corruption. | I `2019.csv` används detta som ett positivt faktorbidrag i WHR-modellen. Därför beskriver vi höga värden i grafer som `Perceived lack of corruption`. |

Tolkningsnotering: den bakomliggande enkätfrågan handlar om uppfattad korruption, men Kaggle-kolumnen är ett bearbetat bidrag till `Score`. `Perceptions of corruption` i `2019.csv` ska därför inte läsas som rå enkätdata.


In [ ]:
import os
import warnings

# Gör UMAP/numba-cache mer robust i miljöer där standardcache inte är skrivbar.
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba-cache")
os.environ.setdefault("KMP_WARNINGS", "FALSE")

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import umap
from scipy.stats import shapiro
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import numpy as np
from numpy.linalg import svd
import plotly.express as px
from IPython.display import display

warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")
warnings.filterwarnings("ignore", message="The library used by the .*country names.*")

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

In [ ]:
NUMERIC_COLS = [
    "Score",
    "GDP per capita",
    "Social support",
    "Healthy life expectancy",
    "Freedom to make life choices",
    "Generosity",
    "Perceptions of corruption",
]

FEATURE_COLS = [
    "GDP per capita",
    "Social support",
    "Healthy life expectancy",
    "Freedom to make life choices",
    "Generosity",
    "Perceptions of corruption",
]

EXPECTED_COLS = ["Overall rank", "Country or region"] + NUMERIC_COLS

# Variabelförklaringar från Kaggle/World Happiness Report 2019.
# Engelska källfraser behålls för att formuleringarna ska ligga nära originalet.
# Featurekolumnerna efter Score är faktorbidrag, inte råvärden. WHR beskriver
# dem som "components explained by six hypothesized underlying determinants"
# och modellen som "illustrative rather than conclusive".
VARIABLE_EXPLANATIONS = {
    "Overall rank": "Rankning i 2019.csv; WHR beskriver detta som ranking of happiness scores.",
    "Country or region": "Land/region i 2019.csv; WHR använder även formuleringen country/territory.",
    "Score": "WHR: nationellt medelvärde för Cantril life ladder, med steg från 0 till 10.",
    "GDP per capita": "Grundvariabel: GDP per capita i purchasing power parity. I 2019.csv visas faktorbidraget från GDP per capita.",
    "Social support": "Grundvariabel: GWP-fråga om relatives or friends you can count on vid problem.",
    "Healthy life expectancy": "Grundvariabel: Healthy life expectancies at birth från WHO Global Health Observatory.",
    "Freedom to make life choices": "Grundvariabel: GWP-frågan om freedom to choose what you do with your life.",
    "Generosity": "Grundvariabel: fråga om donated money to a charity, beräknad som residual efter regression mot GDP per capita.",
    "Perceptions of corruption": "Grundvariabel: medel av två GWP-frågor: Is corruption widespread throughout the government or not? och Is corruption widespread within businesses or not? Vid saknad government-data används business corruption. I 2019.csv visas faktorbidraget, vilket motsvarar freedom from corruption/absence of corruption i WHR:s modell.",
}

DATA_PATH = "2019.csv"


## Läs in och kontrollera data

Först läser vi in `2019.csv` och gör en enkel integritetskontroll: saknade värden, dubbletter, datatyper och negativa värden i de numeriska kolumnerna.

In [ ]:
def check_data_integrity(df):
    problems = []

    null_counts = df.isnull().sum()
    if null_counts.any():
        for col, count in null_counts[null_counts > 0].items():
            problems.append(f"Saknade värden: {count} stycken i '{col}'.")

    duplicates = df.duplicated().sum()
    if duplicates > 0:
        problems.append(f"{duplicates} dubbletter hittades i datasetet.")

    for col in EXPECTED_COLS:
        if col not in df.columns:
            problems.append(f"Kolumnen '{col}' saknas i datasetet.")

    for col in NUMERIC_COLS:
        if col not in df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            problems.append(f"Kolumnen '{col}' är inte numerisk.")
        elif (df[col] < 0).any():
            problems.append(f"Negativa värden hittades i '{col}'.")

    if "Score" in df.columns and pd.api.types.is_numeric_dtype(df["Score"]):
        if not df["Score"].between(0, 10).all():
            problems.append("Score innehåller värden utanför Cantril-skalan 0-10.")

    if "Overall rank" in df.columns and df["Overall rank"].duplicated().any():
        problems.append("Overall rank innehåller dubbletter.")

    if "Country or region" in df.columns and df["Country or region"].duplicated().any():
        problems.append("Country or region innehåller dubbletter.")

    if problems:
        print("Varning: Följande problem identifierades:")
        for problem in problems:
            print(f"- {problem}")
    else:
        print("Datan är intakt och redo för analys: inga saknade värden, dubbletter eller negativa värden.")

    return not problems


df = pd.read_csv(DATA_PATH)
if not check_data_integrity(df):
    raise ValueError("Datasetet klarade inte integritetskontrollen.")
df.head()


In [ ]:
df.info()

## Normalfördelning av `Score`

Vi använder Shapiro-Wilk-testet för att kontrollera om `Score` kan betraktas som normalfördelad.

- **H0:** `Score` är normalfördelad.
- Om p-värdet är minst 0.05 förkastar vi inte H0.

In [ ]:
def hypotes_test(p_value):
    if p_value >= 0.05:
        return "Nollhypotesen förkastas ej."
    return "Nollhypotesen förkastas."


test = shapiro(df["Score"])
print(
    f"Shapiro (H0: Score är normalfördelad)\n"
    f"teststatistik: {test.statistic:.3f}\n"
    f"p-värde: {test.pvalue:.3f}\n"
    f"Resultat: {hypotes_test(test.pvalue)}"
)

## Fördelning av `Score`

Datasetet är sorterat efter `Overall rank`, vilket gör det enkelt att se hur `Score` faller från högst rankade till lägst rankade land.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="Overall rank", y="Score", s=55, color="#2f6f9f", edgecolor="white")
plt.title("Fördelning av Score efter Overall rank", fontsize=14)
plt.xlabel("Overall rank", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Global karta över `Score`

Kartvisualiseringen ger en snabb överblick över hur `Score` varierar geografiskt.

In [ ]:
fig = px.choropleth(
    df,
    locations="Country or region",
    locationmode="country names",
    color="Score",
    color_continuous_scale="RdYlGn",
    title="Global fördelning av Score",
)
fig.show()

## `GDP per capita`, `Score` och Perceived lack of corruption

Visualiseringen undersöker sambandet mellan `GDP per capita` och `Score`. Färg och punktstorlek visar det bearbetade faktorbidraget `Perceived lack of corruption`, vilket i denna CSV motsvaras av kolumnen `Perceptions of corruption`.

Högre värde tolkas som **större faktorbidrag från låg uppfattad korruption / frånvaro av korruption**.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

corruption_values = df["Perceptions of corruption"]
point_sizes = 70 + (corruption_values * 120)
scatter_plot = ax.scatter(
    df["GDP per capita"],
    df["Score"],
    c=corruption_values,
    cmap="Greens",
    vmin=0,
    vmax=corruption_values.max(),
    s=point_sizes,
    alpha=0.8,
    edgecolors="black",
    linewidths=0.6,
)

trend_x = np.linspace(df["GDP per capita"].min(), df["GDP per capita"].max(), 100)
trend_m, trend_b = np.polyfit(df["GDP per capita"], df["Score"], 1)
ax.plot(trend_x, trend_m * trend_x + trend_b, color="black", linestyle="--", linewidth=1.5, label="Linjär trend")

label_df = pd.concat([
    df.nlargest(4, "Perceptions of corruption"),
    df.nsmallest(4, "Perceptions of corruption"),
]).drop_duplicates(subset="Country or region")

for _, row in label_df.iterrows():
    ax.annotate(
        row["Country or region"],
        (row["GDP per capita"], row["Score"]),
        textcoords="offset points",
        xytext=(6, 5),
        fontsize=9,
        fontweight="bold",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1.5),
    )

ax.set_title("Jämförelse mellan GDP per capita, Score och Perceived lack of corruption", fontsize=16, pad=20)
ax.set_xlabel("GDP per capita", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
colorbar = fig.colorbar(scatter_plot, ax=ax)
colorbar.set_label("Perceived lack of corruption (faktorbidrag)")
colorbar.set_ticks(np.linspace(0, corruption_values.max(), 5))
ax.legend(loc="lower right")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## SVD: varför skalning spelar roll

I analysen används SVD för att jämföra hur variablerna bidrar till komponenterna före och efter standardisering.

- Med enbart centrering kan variabler med större variation dominera.
- Med standardisering får alla variabler medelvärde 0 och standardavvikelse 1, vilket gör jämförelsen mer balanserad.

Tabellerna visar absoluta variabelbidrag. Det gör att tecknet i komponentladdningarna inte feltolkas som positivt eller negativt bidrag.

In [ ]:
X = df[FEATURE_COLS].values

# 1. Enbart centrering
X_centered = X - X.mean(axis=0)
U_c, S_c, Vt_c = svd(X_centered, full_matrices=False)
explained_var_c = (S_c**2 / np.sum(S_c**2)) * 100
svd_df_c = pd.DataFrame(np.abs(Vt_c), columns=FEATURE_COLS, index=[f"K{i+1}" for i in range(len(S_c))])

# 2. Standardisering
X_std = (X - X.mean(axis=0)) / X.std(axis=0)
U_s, S_s, Vt_s = svd(X_std, full_matrices=False)
explained_var_s = (S_s**2 / np.sum(S_s**2)) * 100
svd_df_s = pd.DataFrame(np.abs(Vt_s), columns=FEATURE_COLS, index=[f"K{i+1}" for i in range(len(S_s))])

print("SVD: absoluta variabelbidrag (endast centrering)")
display(svd_df_c.iloc[:2].T.round(3))

print("SVD: absoluta variabelbidrag (standardiserad data)")
display(svd_df_s.iloc[:2].T.round(3))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

top_vars_c = svd_df_c.idxmax(axis=1)
top_vars_s = svd_df_s.idxmax(axis=1)
top_text_c = "Största bidrag:\n" + "\n".join([f"{k}: {v}" for k, v in top_vars_c.items()])
top_text_s = "Största bidrag:\n" + "\n".join([f"{k}: {v}" for k, v in top_vars_s.items()])

ax1.bar([f"K{i+1}" for i in range(len(S_c))], explained_var_c, color="skyblue")
ax1.set_title("Förklarad varians (endast centrering)")
ax1.set_ylabel("Förklarad varians (%)")
for i, v in enumerate(explained_var_c):
    ax1.text(i, v + 1, f"{v:.1f}%", ha="center")
ax1.text(
    0.98,
    0.95,
    top_text_c,
    transform=ax1.transAxes,
    va="top",
    ha="right",
    bbox=dict(facecolor="white", edgecolor="gray", alpha=0.85),
)

ax2.bar([f"K{i+1}" for i in range(len(S_s))], explained_var_s, color="salmon")
ax2.set_title("Förklarad varians (standardiserad data)")
for i, v in enumerate(explained_var_s):
    ax2.text(i, v + 1, f"{v:.1f}%", ha="center")
ax2.text(
    0.98,
    0.95,
    top_text_s,
    transform=ax2.transAxes,
    va="top",
    ha="right",
    bbox=dict(facecolor="white", edgecolor="gray", alpha=0.85),
)

plt.tight_layout()
plt.show()

## Varför UMAP?

Vi använder UMAP (Uniform Manifold Approximation and Projection) eftersom målet är att se om länder bildar tydliga grupper baserat på sina lyckoprofiler.

UMAP passar bra i denna analys av tre skäl:

1. **Kluster:** metoden tenderar att dra ihop liknande observationer och separera olika observationer.
2. **Icke-linjära samband:** sociala och ekonomiska variabler följer sällan enkla linjära mönster.
3. **Lokal och global struktur:** UMAP kan både visa vilka länder som liknar varandra och hur större grupper förhåller sig till varandra.

Vi jämför tre varianter: utan skalning, med standardisering och med min-max-normalisering.

In [ ]:
print("Jämförelse av skalningsmetoder inför UMAP:")
print("- Utan skalning: känslig för variabler med stora värden, till exempel GDP per capita")
print("- Standardisering: balanserar variabler och ger ofta tydligare struktur")
print("- Normalisering: kan komprimera variation och ibland ge sämre separation")

In [ ]:
def make_umap_df(X_input, score, countries, random_state=200):
    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="euclidean",
        random_state=random_state,
    )
    embedding = reducer.fit_transform(X_input)
    umap_df = pd.DataFrame(embedding, columns=["UMAP1", "UMAP2"])
    umap_df["Score"] = score.values
    umap_df["Country or region"] = countries.values
    return umap_df


def plot_umap(umap_data, title):
    fig = px.scatter(
        umap_data,
        x="UMAP1",
        y="UMAP2",
        color="Score",
        hover_name="Country or region",
        color_continuous_scale="Viridis",
        title=title,
    )
    fig.update_traces(marker=dict(size=9, opacity=0.85, line=dict(width=1, color="black")))
    fig.update_layout(plot_bgcolor="white", paper_bgcolor="#f7f7f7")
    fig.show()

### UMAP utan skalning

Faktorvärdena används direkt från `2019.csv`, utan extra skalning. Denna version är mest känslig för variabler med större numerisk variation.

In [ ]:
X_raw = df[FEATURE_COLS]
umap_df_raw = make_umap_df(X_raw, df["Score"], df["Country or region"])
plot_umap(umap_df_raw, "UMAP-projektion (utan skalning)")

### UMAP med standardisering

Variablerna standardiseras så att varje variabel får medelvärde 0 och standardavvikelse 1. Det gör att varje variabel får jämförbar vikt i avståndsberäkningen.

In [ ]:
standard_scaler = StandardScaler()
X_standardised = standard_scaler.fit_transform(df[FEATURE_COLS])
umap_df_std = make_umap_df(X_standardised, df["Score"], df["Country or region"])
plot_umap(umap_df_std, "UMAP-projektion (standardiserad data)")

### UMAP med min-max-normalisering

Varje variabel skalas till intervallet `[0, 1]`. Det gör variablerna jämförbara, men kan också komprimera variationen.

In [ ]:
minmax_scaler = MinMaxScaler()
X_normalised = minmax_scaler.fit_transform(df[FEATURE_COLS])
umap_df_norm = make_umap_df(X_normalised, df["Score"], df["Country or region"])
plot_umap(umap_df_norm, "UMAP-projektion (min-max-normaliserad data)")

## Slutsats

Analysen pekar på att `Score` inte kan förstås genom en enda faktor. `GDP per capita` är starkt kopplad till högre `Score`, men visualiseringarna och den standardiserade analysen visar att flera sociala faktorer också är centrala.

För ett datadrivet partiprogram innebär det att ekonomisk utveckling bör balanseras med faktorer som `Social support`, `Healthy life expectancy`, `Freedom to make life choices` och faktorbidrag kopplade till låg uppfattad korruption, visualiserat som `Perceived lack of corruption`.

UMAP används som en karta över lyckoprofiler: den visar inte en enkel rak linje mot lycka, utan flera möjliga vägar och grupper av länder som liknar varandra.